# Week 4: Color Segmentation & Object Tracking

Phase 1: Classical Computer Vision — Part 4 of 5

After detecting shapes using contours last week, this week moves into something more dynamic: **color-based segmentation and tracking**.

The core question for this week:

> “Can a computer follow an object over time?”

Instead of working with static images, this notebook processes **video frames in real time**, tracking a single colored object as it moves.


## Objectives for Week 4

- Segment objects based on color
- Understand why HSV color space is useful
- Create binary masks using color thresholds
- Clean masks using morphological operations
- Track an object’s position across video frames
- Visualize movement using a trajectory


In [1]:
import cv2
import numpy as np
from collections import deque

## Step 1: Reading Video Frames

The first step is working with a video stream (or webcam input).

Each frame is treated as a standalone image, but processed sequentially.

Moving to video introduces:
- Real-time performance considerations
- Immediate visual feedback
- Greater sensitivity to inefficiencies


In [2]:
cap = cv2.VideoCapture(0)  # Use 0 for webcam, or provide video file path

# Store object trajectory
pts = deque(maxlen=64)

## Step 2: Converting to HSV Color Space

Instead of working in BGR, frames are converted to HSV.

HSV separates:
- Hue (color)
- Saturation (color intensity)
- Value (brightness)

This makes isolating specific colors much easier under varying lighting conditions.

In [3]:
ret, frame = cap.read()

frame = cv2.flip(frame, 1)
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

cv2.imshow("Original Frame", frame)
cv2.imshow("HSV Frame", hsv)

cv2.waitKey(0)
cv2.destroyAllWindows()

## Step 3: Color Thresholding (Mask Creation)

Define lower and upper HSV bounds for the target color.
Use `inRange` to create a binary mask.

In [4]:
# Example: blue object
lower_color = np.array([100, 150, 50])
upper_color = np.array([140, 255, 255])

mask = cv2.inRange(hsv, lower_color, upper_color)

cv2.imshow("Mask", mask)
cv2.waitKey(0)
cv2.destroyAllWindows()


## Step 4: Morphological Operations

Clean the mask using erosion and dilation.
This removes small noise blobs and stabilizes segmentation.


In [5]:
kernel = np.ones((5,5), np.uint8)

clean_mask = cv2.erode(mask, kernel, iterations=2)
clean_mask = cv2.dilate(clean_mask, kernel, iterations=2)

cv2.imshow("Clean Mask", clean_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()


## Step 5: Finding the Object's Centroid

Detect contours from the cleaned mask.
Select the largest contour and compute its centroid using image moments.

In [6]:
contours, _ = cv2.findContours(
    clean_mask.copy(),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

if len(contours) > 0:
    largest_contour = max(contours, key=cv2.contourArea)

    if cv2.contourArea(largest_contour) > 500:
        M = cv2.moments(largest_contour)

        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            cv2.circle(frame, (cx, cy), 6, (0, 0, 255), -1)
            cv2.imshow("Centroid Detection", frame)
            cv2.waitKey(0)
            cv2.destroyAllWindows()


## Step 6: Real-Time Tracking and Trajectory Visualization

Now we combine all previous steps into a continuous loop.
Centroids are stored and connected to visualize movement.

In [ ]:
pts = deque(maxlen=64)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(hsv, lower_color, upper_color)

    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    contours, _ = cv2.findContours(
        mask.copy(),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    center = None

    if len(contours) > 0:
        largest_contour = max(contours, key=cv2.contourArea)

        if cv2.contourArea(largest_contour) > 500:
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                center = (
                    int(M["m10"] / M["m00"]),
                    int(M["m01"] / M["m00"])
                )
                cv2.circle(frame, center, 6, (0, 0, 255), -1)

    pts.appendleft(center)

    for i in range(1, len(pts)):
        if pts[i - 1] is None or pts[i] is None:
            continue
        cv2.line(frame, pts[i - 1], pts[i], (0, 255, 0), 2)

    cv2.imshow("Tracking", frame)
    cv2.imshow("Mask", mask)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
